## Clinical Data Preprocessing
Steps include:

1. Cohort definition: filter RA (TACERA), remove (VACCINE) from df_clinical, save as df_clinical_ra_only
2. Raw data audit (df_clinical_ra_only, df_steroids, df_meds) and variance report
3. Individual dataset cleaning (clean each table):
4. a. df_clinical_ra_only: type fixing, standardise binary, missing values, encoding categories (gender, race), keep digest for merging, remove redundancies (inkl. duplicate columns, constant values, highly missing features). Export as df_clinical_ra_only_processed
4. b. df_steroids: aggregate (to a patient level), type fixing, missing values, encoding, etc. Export as df_steroids_processed
4. c. df_meds: aggregate (to a patient level), type fixing, missing values, encoding. Export as df_meds_processed
5. Merge df_clinical_ra_only_processed, df_steroids_processed, df_meds_processed into df_clinical_processed_final (one row per patient, all clinical data in one table)

## Add common imports
From /src/01_common_imports.py and from other relevant /src/ files

In [1]:
from src.common_imports import *
import os

from src.data_audit import data_audit
from src.low_variance_report import low_variance_report

from src.load_data import load_all_sheets

from src.sanity_checks import *

from src.cleaning_clinical import *

from src.config.data_contract_preprocessing import CLINICAL_CONTRACT, STEROIDS_CONTRACT, MEDS_CONTRACT

Loading Libraries
Done loading


## Load All Sheets and extract df_clinical

In [2]:
data = load_all_sheets()
df_clinical = data["df_clinical"]

Loading clinical...
Loading protogen...
Loading somascan...
Done loading


## Remove non-RA patients (those in the VACCINE and not in the TACERA study) and export as new dataframe

In [3]:
df_clinical_ra_only = filter_ra_cohort(df_clinical)
df_clinical_ra_only["Study"].value_counts()


===== RA COHORT FILTER =====
Before filtering: 327 rows
After filtering:  275 rows
Removed (non-RA / other cohorts): 52



Study
TACERA    275
Name: count, dtype: int64

## Clinical Data Audit (RA-only patients) / Report per DataFrame
df_clinical

In [4]:
data_audit(df_clinical_ra_only, name="clinical dataset")


DATA AUDIT: CLINICAL DATASET

1. STRUCTURE
Shape: 275 rows × 67 columns

2. DUPLICATES
Duplicate rows: 0

3. MISSING VALUES

Columns > 40% missing:
                             missing_count  missing_pct
Hep B serology wk 9 (IU/mL)            275          1.0
vaccine centre                         275          1.0

Moderate missing values:
                    missing_count  missing_pct
FinalxRAYScore                 40     0.145455
HEIGHT                          7     0.025455
InitialxRAYScore                3     0.010909
Erosive                         3     0.010909
Remission month                 2     0.007273
Remission(<2.6DAS)              2     0.007273
HighDisease(>4DAS)              2     0.007273
CRP.9M                          1     0.003636

4. NUMERIC FEATURES
Numeric columns: 3

Top Outliers by IQR and 3σ:
                mean       std  min   max  outliers_iqr  outliers_3s
Region      4.403636  3.341946  1.0  10.0             0            0
Hub         3.454545  1.866

{'high_missing':                              missing_count  missing_pct
 Hep B serology wk 9 (IU/mL)            275          1.0
 vaccine centre                         275          1.0,
 'mid_missing':                     missing_count  missing_pct
 FinalxRAYScore                 40     0.145455
 HEIGHT                          7     0.025455
 InitialxRAYScore                3     0.010909
 Erosive                         3     0.010909
 Remission month                 2     0.007273
 Remission(<2.6DAS)              2     0.007273
 HighDisease(>4DAS)              2     0.007273
 CRP.9M                          1     0.003636}

df_steroids

In [5]:
df_steroids = data["df_steroids"]
data_audit(df_steroids, name="clinical steroids dataset")


DATA AUDIT: CLINICAL STEROIDS DATASET

1. STRUCTURE
Shape: 414 rows × 10 columns

2. DUPLICATES
Duplicate rows: 0

3. MISSING VALUES

Columns > 40% missing:
Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []

Moderate missing values:
                    missing_count  missing_pct
Date of Assessment              2     0.004831

4. NUMERIC FEATURES
Numeric columns: 4

Top Outliers by IQR and 3σ:
                      mean        std   min     max  outliers_iqr  outliers_3s
Joint Injected   17.495169   3.959080   3.0    29.0            93           12
Unit              1.009662   0.120143   1.0     3.0             3            3
Dose            100.454831  73.450490 -99.0  1000.0             9            2
Assessment        6.371981   4.282041   3.0    19.0             1            0

5. CATEGORICAL CONSISTENCY
Columns with inconsistent casing: ['Digest', '4. Has the patient received a steroid injection?', 'Steroid', 'Route']

6. CLEANING SUMMARY
- High missing (> 40%): 0
- 

{'high_missing': Empty DataFrame
 Columns: [missing_count, missing_pct]
 Index: [],
 'mid_missing':                     missing_count  missing_pct
 Date of Assessment              2     0.004831}

df_meds

In [6]:
df_meds = data["df_meds"]
data_audit(df_meds, name="clinical meds dataset")


DATA AUDIT: CLINICAL MEDS DATASET

1. STRUCTURE
Shape: 2534 rows × 9 columns

2. DUPLICATES
Duplicate rows: 0

3. MISSING VALUES

Columns > 40% missing:
Empty DataFrame
Columns: [missing_count, missing_pct]
Index: []

Moderate missing values:
                    missing_count  missing_pct
Dose                            6     0.002368
Route                           6     0.002368
Frequency                       6     0.002368
Date of Assessment              5     0.001973

4. NUMERIC FEATURES
Numeric columns: 3

Top Outliers by IQR and 3σ:
                  mean         std  min     max  outliers_iqr  outliers_3s
Dose        126.882516  258.375293  1.0  2500.0           156          113
Unit          1.004736    0.097224  1.0     3.0             6            6
Assessment    8.838595    4.910288  3.0    19.0             0            0

5. CATEGORICAL CONSISTENCY
Columns with inconsistent casing: ['Digest', 'RA Medication', 'Frequency', 'Route']

6. CLEANING SUMMARY
- High missing (> 4

{'high_missing': Empty DataFrame
 Columns: [missing_count, missing_pct]
 Index: [],
 'mid_missing':                     missing_count  missing_pct
 Dose                            6     0.002368
 Route                           6     0.002368
 Frequency                       6     0.002368
 Date of Assessment              5     0.001973}

## Clinical Low Variance Audit / Report per DataFrame
df_clinical

In [7]:
low_variance_report(df_clinical_ra_only, name="dataset")


===== LOW VARIANCE REPORT: DATASET =====

Total numeric features: 3
Low variance threshold: 0.01
Low variance features found: 0

No low-variance features detected.


{'low_variance_features': Series([], dtype: float64),
 'variance': Hub            3.482415
 Region        11.168600
 REGION_HUB    12.398089
 dtype: float64,
 'drop_columns': []}

df_steroids

In [8]:
low_variance_report(df_steroids, name="dataset")


===== LOW VARIANCE REPORT: DATASET =====

Total numeric features: 4
Low variance threshold: 0.01
Low variance features found: 0

No low-variance features detected.


{'low_variance_features': Series([], dtype: float64),
 'variance': Unit                 0.014434
 Joint Injected      15.674311
 Assessment          18.335872
 Dose              5394.974444
 dtype: float64,
 'drop_columns': []}

df_meds

In [9]:
low_variance_report(df_meds, name="dataset")


===== LOW VARIANCE REPORT: DATASET =====

Total numeric features: 3
Low variance threshold: 0.01
Low variance features found: 1

Top low-variance features:
Unit    0.009452
dtype: float64


{'low_variance_features': Unit    0.009452
 dtype: float64,
 'variance': Unit              0.009452
 Assessment       24.110930
 Dose          66757.792286
 dtype: float64,
 'drop_columns': ['Unit']}

## Clean clinical data set (patient level)

In [10]:
df_clinical_clean = clean_patient_level_data(df_clinical_ra_only, CLINICAL_CONTRACT)
print(df_clinical_clean.head())

sanity_check_clinical(df_clinical_clean, CLINICAL_CONTRACT)

  Patient_ID                                             Digest  Region  Hub  \
0    TAC1000  B1895B6B8F7B325B977C180A25C74D079EFB5EB306693D...     1.0  1.0   
1    TAC1001  59749E6B6785A9983100DA76AB0AE9484B52B2422AB362...     1.0  1.0   
2    TAC1002  9C459B98832CB4E81C90B843BF7A7925D5E01A1A306586...     1.0  1.0   
3    TAC1003  308098EB685FC45FA0CA61C88D25D810E16C9DADBBA77C...     1.0  1.0   
4    TAC1004  3A9C04CAA9BBAAE16A20614D827FA3B55AA09C94F69284...     1.0  1.0   

   REGION_HUB   Study ACPA.POSITIVE RHUEMATOID.FACTOR  DAS28.0M  DAS28.3M  \
0         1.0  TACERA             1                 1      7.27      1.91   
1         1.0  TACERA             1                 1      7.83      5.37   
2         1.0  TACERA             1                 1      3.43      0.77   
3         1.0  TACERA             1                 1      5.76      3.59   
4         1.0  TACERA             1                 1      6.52      2.13   

   ...   AGE                                            

## Clean steroid data set (event level)

In [11]:
#df_steroids_clean = clean_raw_artifacts(df_steroids, STEROIDS_CONTRACT)
df_steroids_clean = clean_event_level_data(df_steroids, STEROIDS_CONTRACT)
print(df_steroids_clean.head())

(df_steroids["Dose"] == -99).sum()

print(df_steroids_clean["Dose"].describe())
#sanity_structure(df_steroids_clean)
#sanity_contract_check(df_steroids_clean, STEROIDS_CONTRACT)
#sanity_numeric(df_steroids_clean, STEROIDS_CONTRACT)
#sanity_binary(df_steroids_clean, STEROIDS_CONTRACT)
#sanity_categorical(df_steroids_clean, STEROIDS_CONTRACT)
#sanity_dates(df_steroids_clean, STEROIDS_CONTRACT)
#sanity_duplicates(df_steroids_clean, ["Digest", "Date Given", "Steroid"])
#df_steroids_clean.head(20)

#df_steroids_clean.groupby(
#    ["Digest", "Date Given", "Steroid", "Dose", "Route", "Joint Injected"]
#).size().sort_values(ascending=False).head(20)

                                              Digest Date of Assessment  \
0  59749E6B6785A9983100DA76AB0AE9484B52B2422AB362...         2013-02-05   
1  59749E6B6785A9983100DA76AB0AE9484B52B2422AB362...         2013-02-05   
2  59749E6B6785A9983100DA76AB0AE9484B52B2422AB362...         2013-02-05   
3  9C459B98832CB4E81C90B843BF7A7925D5E01A1A306586...         2013-07-23   
4  9C459B98832CB4E81C90B843BF7A7925D5E01A1A306586...         2013-07-23   

   4. Has the patient received a steroid injection?  Assessment  \
0                                                 1           3   
1                                                 1           3   
2                                                 1           3   
3                                                 1           3   
4                                                 1           3   

                      Steroid   Dose Unit Date Given Joint Injected  \
0  Methylprednisolone Acetate  120.0    1 2013-02-05             19   
1  M

## Clean meds data set (event level)

In [12]:
#df_meds_clean = clean_raw_artifacts(df_meds, STEROIDS_CONTRACT)
df_meds_clean = clean_event_level_data(df_meds, MEDS_CONTRACT)

print(df_meds_clean.head())

sanity_structure(df_meds_clean)
sanity_contract_check(df_meds_clean, MEDS_CONTRACT)
sanity_numeric(df_meds_clean, MEDS_CONTRACT)
sanity_categorical(df_meds_clean, MEDS_CONTRACT)
sanity_binary(df_meds_clean, MEDS_CONTRACT)
sanity_dates(df_meds_clean, MEDS_CONTRACT)
sanity_duplicates(df_meds_clean, subset=["Digest", "Date Started", "RA Medication"])

df_meds_clean["Assessment"].value_counts().head(10)
df_meds_clean["Unit"].value_counts().head(10)

df_meds_clean[df_meds_clean.duplicated(keep=False)].sort_values(
    ["Digest", "Date Started"]
)

                                              Digest Date of Assessment  \
0  B1895B6B8F7B325B977C180A25C74D079EFB5EB306693D...         2013-04-29   
1  B1895B6B8F7B325B977C180A25C74D079EFB5EB306693D...         2013-04-29   
2  B1895B6B8F7B325B977C180A25C74D079EFB5EB306693D...         2013-04-29   
3  B1895B6B8F7B325B977C180A25C74D079EFB5EB306693D...         2013-07-18   
4  B1895B6B8F7B325B977C180A25C74D079EFB5EB306693D...         2013-07-18   

   Assessment       RA Medication   Dose  Unit    Frequency Route Date Started  
0           3        Methotrexate   20.0     1       Weekly  Oral   2013-03-18  
1           3  Hydroxychloroquine  200.0     1  Twice daily  Oral   2013-05-02  
2           3          Folic Acid    5.0     1       Weekly  Oral   2013-05-02  
3           6  Hydroxychloroquine  200.0     1  Twice daily  Oral   2013-05-02  
4           6        Methotrexate   20.0     1       Weekly  Oral   2013-03-18  
Shape: (2534, 9)

Dtypes:
 Digest                           str

,Digest,Date of Assessment,Assessment,RA Medication,Dose,Unit,Frequency,Route,Date Started


## Aggregate event-level data (steroids, meds) to patient level (clinical) and merge with clinical data to create final clinical dataset for analysis

In [13]:
df_steroids_agg = aggregate_steroids(df_steroids_clean)
print(df_steroids_agg.head())
df_meds_agg = aggregate_meds(df_meds_clean)
print(df_meds_agg.head())

sanity_structure(df_steroids_agg)
sanity_structure(df_meds_agg)

check_unique_patients(df_steroids_agg)
check_unique_patients(df_meds_agg)

                                              Digest  steroid_injection_count  \
0  0110A6D8FE1F42B8F690C418F4A76379A3D95E0B68E625...                        1   
1  016BBC0766785143D12B9EEE5B3B9DDA4C0C850D720120...                        3   
2  036A2E585D2906AA2EE0114C9F34B066DCF0C9107F4AF0...                        2   
3  038B1596C82BBB7DA336C71FBAE7B9DE285D27B258430C...                        2   
4  040AC57AA2EF267C057421013A9D2762F9EDF5078CEC1C...                        3   

   total_dose   mean_dose  max_dose  intraarticular_count  \
0        40.0   40.000000      40.0                     0   
1       360.0  120.000000     120.0                     0   
2       240.0  120.000000     120.0                     0   
3       240.0  120.000000     120.0                     0   
4       200.0   66.666667      80.0                     1   

   intramuscular_count               first_injection last_injection  
0                    0 2014-02-19 00:00:00.000000000     2014-02-19  
1     

## Merge all clinical data into one final dataframe (one row per patient, all clinical data in one table)

In [14]:
df_final_merged = merge_patient_datasets(df_clinical_clean, df_steroids_agg, df_meds_agg)

print(df_final_merged.shape)
print(df_final_merged["Digest"].nunique())
print(len(df_final_merged))
df_final_merged["Digest"].duplicated().sum() #Must be 0 for a successful merge to patient level
print(df_final_merged.head())

(275, 71)
275
275
  Patient_ID                                             Digest  Region  Hub  \
0    TAC1000  B1895B6B8F7B325B977C180A25C74D079EFB5EB306693D...     1.0  1.0   
1    TAC1001  59749E6B6785A9983100DA76AB0AE9484B52B2422AB362...     1.0  1.0   
2    TAC1002  9C459B98832CB4E81C90B843BF7A7925D5E01A1A306586...     1.0  1.0   
3    TAC1003  308098EB685FC45FA0CA61C88D25D810E16C9DADBBA77C...     1.0  1.0   
4    TAC1004  3A9C04CAA9BBAAE16A20614D827FA3B55AA09C94F69284...     1.0  1.0   

   REGION_HUB   Study ACPA.POSITIVE RHUEMATOID.FACTOR  DAS28.0M  DAS28.3M  \
0         1.0  TACERA             1                 1      7.27      1.91   
1         1.0  TACERA             1                 1      7.83      5.37   
2         1.0  TACERA             1                 1      3.43      0.77   
3         1.0  TACERA             1                 1      5.76      3.59   
4         1.0  TACERA             1                 1      6.52      2.13   

   ...  intraarticular_count  intramus

## Clean the final merged dataset and export for analysis

In [15]:
FINAL_CONTRACT = {
    "id_columns": ["Digest"]
}

df_final_merged_cleaned = final_preprocessing_full_dataset(df_final_merged, FINAL_CONTRACT)

df_final_merged_cleaned.to_parquet("../mid_processing_datasets/clinical_merged.parquet")
